# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

### Clasificación de las piezas

- Datos históricos de entrenamiento:
  fueron utilizados previamente para entrenar el modelo. No se utilizan
  directamente durante la inferencia.

- Artefacto del modelo:
  es el modelo ya entrenado proporcionado por el profesor. Recibe el vector
  de 11 características preprocesadas y devuelve una predicción.

- Datos nuevos de inferencia:
  son las filas de `inference_samples.csv`. Primero deben validarse contra
  el contrato y después convertirse al vector numérico esperado por el modelo.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [1]:
from dataclasses import dataclass
import csv
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = (
    Path.cwd().parents[5]
    / "assets"
    / "03-wine-quality"
    / "inference_samples.csv"
)
print(DATA_PATH)
print(DATA_PATH.exists())

c:\Users\mygar\Desktop\Master ICAI 2026-27\Operación de Modelos\icai-muiaap-operacion-de-modelos\semana3\assets\03-wine-quality\inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [13]:
# TODO: implementa la lectura del CSV.
# Pista: usa DATA_PATH.open(newline="", encoding="utf-8")
rows = []
row = {}

with DATA_PATH.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

row = rows[0]

print(len(rows))
print(row.keys())
print()



5
dict_keys(['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol'])



### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo? sample_id
2. ¿Cuántas características numéricas hay? 11
3. ¿Qué columna extra añadirías para probar el contrato? Cantidad de la muestra, el contrato debería rechazarla
4. ¿Qué valor de `ph` sería claramente inválido? -4

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Tipo | Unidad | Rango | ¿Nulo? |
|---|---|---|---|---|
| `fixed_acidity` | float | g/dm³ | 0–20 | No |
| `volatile_acidity` | float | g/dm³ | 0–2 | No |
| `citric_acid` | float | g/dm³ | 0–2 | No |
| `residual_sugar` | float | g/dm³ | 0–20 | No |
| `chlorides` | float | g/dm³ | 0–1 | No |
| `free_sulfur_dioxide` | float | mg/dm³ | 0–100 | No |
| `total_sulfur_dioxide` | float | mg/dm³ | 0–300 | No |
| `density` | float | g/cm³ | 0.98–1.01 | No |
| `ph` | float | — | 2.5–4.5 | No |
| `sulphates` | float | g/dm³ | 0–3 | No |
| `alcohol` | float | % vol. | 5–20 | No |

In [3]:
class WineQualityRequestPydantic(BaseModel):
    # TODO: prohíbe campos extra con model_config.
    # TODO: declara las 11 características con Field(ge=..., le=...).
    model_config = ConfigDict(extra="forbid")
    fixed_acidity: float = Field(ge=0.0, le=20.0)
    volatile_acidity: float = Field(ge=0.0, le=2.0)
    citric_acid: float = Field(ge=0.0, le=2.0)
    residual_sugar: float = Field(ge=0.0, le=20.0)
    chlorides: float = Field(ge=0.0, le=1.0)
    free_sulfur_dioxide: float = Field(ge=0.0, le=100.0)
    total_sulfur_dioxide: float = Field(ge=0.0, le=300.0)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0.0, le=3.0)
    alcohol: float = Field(ge=5.0, le=20.0)

# TODO: separa sample_id de la fila y valida solo las características del modelo.
model_row = {
    key: value
    for key, value in row.items()
    if key != "sample_id"
}
request = WineQualityRequestPydantic.model_validate(model_row)

### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

In [4]:
# TODO: completa los casos y captura ValidationError.
try:
    request = WineQualityRequestPydantic.model_validate(model_row)
    print("Fila válida")
    # No se espera error
except ValidationError as error:
    print(error)


extra_row = model_row.copy()
extra_row["unexpected_field"] = 123

try:
    WineQualityRequestPydantic.model_validate(extra_row)
    # Se espera error por campo extra
except ValidationError as error:
    print(error)


invalid_ph_row = model_row.copy()
invalid_ph_row["ph"] = 8.0

try:
    WineQualityRequestPydantic.model_validate(invalid_ph_row)
    # Se espera error por valor fuera de rango
except ValidationError as error:
    print(error)

Fila válida
1 validation error for WineQualityRequestPydantic
unexpected_field
  Extra inputs are not permitted [type=extra_forbidden, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
1 validation error for WineQualityRequestPydantic
ph
  Input should be less than or equal to 4.5 [type=less_than_equal, input_value=8.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [ ]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    # TODO: declara las mismas 11 características.
    fixed_acidity: float 
    volatile_acidity: float 
    citric_acid: float 
    residual_sugar: float 
    chlorides: float 
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float 
    density: float 
    ph: float 
    sulphates: float 
    alcohol: float 

# TODO: crea una instancia válida y prueba qué ocurre con ph=99.0.
valid_dataclass = WineQualityRequestDataclass(**request.model_dump())

invalid_dataclass = WineQualityRequestDataclass(**{**request.model_dump(), "ph": 99.0})
# No da error porque dataclasses no valida los valores de los campos, solo los tipos.

### Decisión de diseño

Completa la frase:

> Usaría Pydantic en la entrada y salida de datos porque permite validar tipos, rangos y detectar campos inesperados. Usaría `dataclass` en procesos internos con datos que ya han sido validados porque es rápido y sencillo pero no valida completamente los datos.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [9]:
# TODO: rellena el orden exacto a partir del CSV y del contrato.
FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol"
)

# TODO: construye vector usando getattr(request, name).
vector = [getattr(request, name) for name in FEATURE_NAMES]
print(vector)

[7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]


### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores? Si
- ¿El clasificador detectaría necesariamente el error? No, el modelo no sabe qué significado tiene cada posición. Interpretaría el valor de alcohol como si fuese density, y viceversa.
- ¿Cómo evitarías que cada script definiera un orden distinto? Definiendo FEATURE_NAMES una sola vez en un lugar común y usando siempre esa constante para construir el vector.

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado? 
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [ ]:
class WineQualityPredictionPydantic(BaseModel):
    # TODO: declara los campos de salida y sus restricciones.
    model_config = ConfigDict(extra="forbid")
    sample_id: str
    quality_band: int
    confidence: float = Field(ge=0.0, le=1.0)
    model_version: str
    preprocessing_version: str

# TODO: escribe un ejemplo de salida para red-001.
prediction = WineQualityPredictionPydantic(
    sample_id="red-001",
    quality_band=5,
    confidence=0.87,
    model_version="1.0",
    preprocessing_version="1.0",
)
print(prediction.model_dump())

{'sample_id': 'red-001', 'predicted_class': 5, 'confidence': 0.87, 'model_version': '1.0', 'preprocessing_version': '1.0'}


### Criterios de aceptación

1. Una fila válida debe poder validarse y transformarse en un vector de
   exactamente 11 valores siguiendo `FEATURE_NAMES`.

2. Una entrada con columnas desconocidas o valores fuera de rango debe
   ser rechazada antes de llamar al modelo.

3. La salida debe incluir la categoría predicha, la confianza,
   la versión del modelo y la versión del preprocesado.

## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Entrega
```text
Entrega una captura o exportación con:

- La tabla de contrato: Realizada en el notebook
- Las respuestas de las prácticas A, B y C: Realizadas en el notebook
- El orden de las 11 características: Realizado en el notebook
- Un ejemplo de entrada inválida: Realizado en el notebook
- Un contrato de salida: Realizado en el notebook
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`:

`contracts.py`: incluiria los contratos de entrada (el de pydantic) y salido
`preprocess.py`: definicion del vector ordenado
`inference.py`:  carga y usa el modelo y realiza la prediccion
`predict_file.py`: lee el csv, valida los campos, predice y genera la salida. Une los anteriores procesos
```